# Chapter 13: the rate table and its five checks

A rate is a row with a start, an end, a source and a checker. This notebook loads the State S table,
shows a rule change on 1 October 2026 and runs the five checks of 13.4 against the synthetic dispatches
and a deliberately broken copy of the table. State S is synthetic; no real rate appears here.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2
import calculator as calc

In [ ]:
base = calc.load_rates()
change = calc.load_rates('rate_table_state_s_oct_change.csv')
change[change.kind == 'duty'][['row_id', 'state', 'band', 'design', 'share', 'floor_per_bl', 'effective_from', 'effective_to', 'notification', 'checked_by', 'checked_on', 'status']]

In [ ]:
sep30 = calc.duty_for(calc.row_in_force(change, 'S', 5.3, '2026-09-30'), 982.80, 7.8, 5.3)
oct1 = calc.duty_for(calc.row_in_force(change, 'S', 5.3, '2026-10-01'), 982.80, 7.8, 5.3)
print('30 September:', inr(sep30), '; 1 October:', inr(oct1), '; difference', r2(oct1 - sep30), 'a case')

## The five checks on the synthetic dispatches

In [ ]:
facts = pd.read_csv('../data/dispatches_state_s.csv', parse_dates=['dispatch_date'])
missing = calc.check_rate_exists(base, facts)
gaps = calc.check_gap_overlap(base[base.state == 'S'], '2026-04-01', '2026-09-30')
recompute = calc.check_recompute(base, facts)
bands = calc.check_band(base, facts)
expired = calc.check_expiry(base, '2026-11-03')
print('1 rate exists:', missing or 'pass')
print('2 no gap, no overlap:', gaps or 'pass')
print('3 recompute:', recompute)
print('4 band:', bands)
print('5 expiry (90 days):', expired or 'pass')

## The same checks on a broken table
Two rows overlap in October, the mild row stops in August and one row was last checked in June.

In [ ]:
broken = change.copy()
broken.loc[broken.row_id == 'S-STRONG-2026-04', 'effective_to'] = pd.Timestamp('2026-10-15')
broken.loc[broken.row_id == 'S-MILD-2026-04', 'effective_to'] = pd.Timestamp('2026-08-31')
broken.loc[broken.row_id == 'S-FEE-REG-2026-04', 'checked_on'] = '2026-06-30'
problems = calc.check_gap_overlap(broken[broken.state == 'S'], '2026-04-01', '2026-10-31')
summary = pd.DataFrame(problems, columns=['state', 'band', 'day', 'problem']).groupby(['band', 'problem']).day.agg(['min', 'max', 'count'])
print(summary)
print('expired:', calc.check_expiry(broken, '2026-11-03'))
try:
    calc.row_in_force(broken, 'S', 5.3, '2026-10-05')
except calc.NoRate as e:
    print('rate exists check:', e)

## What the band check is worth
Apply the mild rate to the 5.3 percent lager on 1,000 cases.

In [ ]:
short_case = r2(1081.08 - 786.24)
short_1000 = r2(1000 * short_case)
print('INR', short_case, 'a case;', inr(short_1000), 'on 1,000 cases')

## The depot walk of 13.2

In [ ]:
opening, receipts, issues = 2000, 6000, 4800
closing = opening + receipts - issues
cover_days = closing / (issues / 30)
sell_in_ratio = receipts / issues
print('closing', closing, '; cover', cover_days, 'days; sell-in', sell_in_ratio, 'times issues')